# Estudo de Caso 6.1 — Percolação em Terreno com Textura Conhecida

**Capítulo Associado:** Capítulo 6 — Percolação em Meio Poroso  
**Nível:** Graduação/Pós-Graduação  
**Tempo estimado:** 60 minutos  
**Autor:** Jader Lugon Junior

---

## 📋 Resumo

Neste estudo de caso, calculamos o perfil de potencial matricial $\psi(z)$ e conteúdo de água $\theta(z)$ em um solo franco-arenoso, considerando:

1. Textura do solo: franco-arenosa (70% areia, 20% silte, 10% argila)
2. Profundidade do lençol freático: $L = 3,0$ m abaixo da superfície
3. Condição de superfície: infiltração constante $q_0 = 1,0 \times 10^{-6}$ m/s
4. Parâmetros de van Genuchten estimados pelo Rosetta

**Objetivo:** Verificar se o fluxo prescrito $q_0$ é compatível com as propriedades hidráulicas do solo.

---

## 🎯 Objetivos de Aprendizagem

- Aplicar a equação de Richards em regime permanente
- Resolver numericamente o perfil de potencial matricial
- Interpretar a influência de $K_{sat}$ e $L$ no perfil de umidade
- Analisar sensibilidade dos parâmetros hidráulicos

---

## 🔧 Requisitos

- Bibliotecas: `numpy`, `scipy`, `matplotlib`
- Conhecimento prévio: Lei de Darcy, modelo de van Genuchten (Cap. 6)

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Ambiente configurado com sucesso!")

---

## 📊 Seção 1: Descrição do Problema

### Parâmetros do Solo (Rosetta)

Para um solo franco-arenoso (70% areia, 20% silte, 10% argila) com $\rho_b = 1,5$ g/cm³, o Rosetta estima:

| Parâmetro | Símbolo | Valor | Unidade |
|-----------|---------|-------|----------|
| Conteúdo residual | $\theta_r$ | 0,058 | m³/m³ |
| Conteúdo saturado | $\theta_s$ | 0,41 | m³/m³ |
| Parâmetro de escala | $\alpha$ | 7,4 | m⁻¹ |
| Parâmetro de forma | $n$ | 1,56 | adimensional |
| Condutividade saturada | $K_{sat}$ | 5,1×10⁻⁵ | m/s |
| Conectividade | $L$ | 0,5 | adimensional |

In [ ]:
# ============================================================
# PARÂMETROS DO SOLO
# ============================================================

# Parâmetros de van Genuchten (Rosetta)
theta_r = 0.058    # m³/m³
theta_s = 0.41     # m³/m³
alpha = 7.4        # m⁻¹
n = 1.56
m = 1 - 1/n
K_sat = 5.1e-5     # m/s (18,36 cm/h = 4,4 m/dia)
L_param = 0.5      # conectividade

# Condições do problema
q0 = 1.0e-6        # m/s (infiltração constante)
L_lençol = 3.0     # m (profundidade do lençol freático)

print("=" * 60)
print("PARÂMETROS DO SOLO FRANCO-ARENOSO")
print("=" * 60)

print(f"\n📊 Parâmetros de van Genuchten:")
print(f"  • θ_r = {theta_r} m³/m³")
print(f"  • θ_s = {theta_s} m³/m³")
print(f"  • α = {alpha} m⁻¹")
print(f"  • n = {n}")
print(f"  • m = 1 - 1/n = {m:.3f}")
print(f"  • K_sat = {K_sat:.2e} m/s ({K_sat*86400:.1f} m/dia)")

print(f"\n🌧️ Condições do problema:")
print(f"  • Infiltração: q₀ = {q0:.2e} m/s ({q0*86400*1000:.1f} mm/dia)")
print(f"  • Lençol freático: L = {L_lençol} m")

print(f"\n✅ Verificação:")
print(f"  • q₀/K_sat = {q0/K_sat:.3f} ({q0/K_sat*100:.1f}%)")
if q0 < K_sat:
    print(f"  • q₀ < K_sat → fluxo compatível com o solo ✓")
else:
    print(f"  • q₀ > K_sat → fluxo INCOMPATÍVEL (solo saturaria)")

---

## 🧮 Seção 2: Modelo de van Genuchten-Mualem

### Funções θ(ψ) e K(θ)

In [ ]:
# ============================================================
# FUNÇÕES DE VAN GENUCHTEN
# ============================================================

def van_genuchten_theta(psi, theta_r, theta_s, alpha, n):
    """Conteúdo de água θ em função do potencial matricial ψ."""
    m = 1 - 1/n
    psi = np.atleast_1d(psi)
    theta = np.zeros_like(psi)
    
    # Para ψ >= 0 (saturado)
    theta[psi >= 0] = theta_s
    
    # Para ψ < 0 (não-saturado)
    mask = psi < 0
    psi_neg = np.abs(psi[mask])
    theta[mask] = theta_r + (theta_s - theta_r) / (1 + (alpha * psi_neg)**n)**m
    
    return theta

def van_genuchten_K(theta, theta_r, theta_s, K_sat, n, L=0.5):
    """Condutividade hidráulica K em função do conteúdo de água θ."""
    m = 1 - 1/n
    theta = np.atleast_1d(theta)
    Se = (theta - theta_r) / (theta_s - theta_r)
    Se = np.clip(Se, 0, 1)
    
    K = K_sat * Se**L * (1 - (1 - Se**(1/m))**m)**2
    
    return K

def K_de_psi(psi, theta_r, theta_s, alpha, n, K_sat, L=0.5):
    """Condutividade hidráulica K em função do potencial matricial ψ."""
    theta = van_genuchten_theta(psi, theta_r, theta_s, alpha, n)
    return van_genuchten_K(theta, theta_r, theta_s, K_sat, n, L)

print("✅ Funções de van Genuchten implementadas!")

---

## 🔬 Seção 3: Solução Numérica do Perfil ψ(z)

### Equação de Richards em Regime Permanente

Para fluxo vertical descendente constante $q = -q_0$ (com $q_0 > 0$):

$$
-q_0 = -K(\psi) \cdot \left(\frac{d\psi}{dz} + 1\right)
$$

Isolando o gradiente de potencial:

$$
\frac{d\psi}{dz} = \frac{q_0}{K(\psi)} - 1
$$

### Esquema Numérico Implícito

Discretizando por diferenças finitas para o nó $i$ (com $z_i > z_{i-1}$):

$$
\frac{\psi_i^{n+1} - \psi_{i-1}^{n+1}}{\Delta z} = \frac{q_0}{K(\psi_i^{n+1})} - 1
$$

A não-linearidade em $K(\psi)$ é tratada por iteração de Picard:

$$
\psi_i^{n+1,(k+1)} = \psi_{i-1}^{n+1} + \Delta z \cdot \left( \frac{q_0}{K(\psi_i^{n+1,(k)})} - 1 \right)
$$

In [ ]:
# ============================================================
# SOLUÇÃO NUMÉRICA DO PERFIL ψ(z)
# ============================================================

print("=" * 60)
print("SOLUÇÃO NUMÉRICA DO PERFIL ψ(z)")
print("=" * 60)

# Discretização
N = 31  # número de nós
dz = L_lençol / (N - 1)  # espaçamento vertical
z = np.linspace(0, -L_lençol, N)  # coordenadas verticais (negativas para baixo)

print(f"\n📊 Discretização:")
print(f"  • Número de nós: N = {N}")
print(f"  • Espaçamento: Δz = {dz:.3f} m")
print(f"  • Profundidade: 0 a {-L_lençol} m")

# Condição de contorno: ψ = 0 no lençol freático (z = -L)
psi = np.zeros(N)
psi[-1] = 0.0  # ψ = 0 no lençol

# Iteração de Picard (de baixo para cima)
tol = 1e-6
max_iter = 100

for i in range(N-2, -1, -1):  # de N-2 até 0
    psi_prev = psi[i+1]  # valor do nó inferior (já calculado)
    
    # Estimativa inicial
    psi_i = psi_prev - dz  # aproximação linear
    
    # Iteração de Picard
    for k in range(max_iter):
        K_i = K_de_psi(psi_i, theta_r, theta_s, alpha, n, K_sat, L_param)
        
        # Evitar divisão por zero
        if K_i < 1e-15:
            K_i = 1e-15
        
        # Atualização
        psi_novo = psi_prev + dz * (q0 / K_i - 1)
        
        # Critério de convergência
        if abs(psi_novo - psi_i) < tol:
            psi[i] = psi_novo
            break
        
        psi_i = psi_novo
    else:
        print(f"  ⚠️ Picard não convergiu no nó {i}")

# Calcular conteúdo de água θ(z)
theta = van_genuchten_theta(psi, theta_r, theta_s, alpha, n)

print(f"\n✅ Solução convergiu!")
print(f"  • ψ na superfície (z=0): {psi[0]:.3f} m")
print(f"  • θ na superfície (z=0): {theta[0]:.3f} m³/m³")
print(f"  • ψ no lençol (z=-{L_lençol}m): {psi[-1]:.3f} m")
print(f"  • θ no lençol (z=-{L_lençol}m): {theta[-1]:.3f} m³/m³")

---

## 📈 Seção 4: Visualização dos Resultados

In [ ]:
# ============================================================
# VISUALIZAÇÃO DOS RESULTADOS
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico 1: Perfil ψ(z)
axes[0].plot(psi, z, 'b-', linewidth=2)
axes[0].set_xlabel('Potencial Matricial, ψ [m]', fontsize=12)
axes[0].set_ylabel('Profundidade, z [m]', fontsize=12)
axes[0].set_title('Perfil de Potencial Matricial', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='r', linestyle='--', label='Superfície')
axes[0].axhline(y=-L_lençol, color='g', linestyle='--', label='Lençol Freático')
axes[0].legend(fontsize=10)

# Gráfico 2: Perfil θ(z)
axes[1].plot(theta, z, 'r-', linewidth=2)
axes[1].set_xlabel('Conteúdo de Água, θ [m³/m³]', fontsize=12)
axes[1].set_ylabel('Profundidade, z [m]', fontsize=12)
axes[1].set_title('Perfil de Conteúdo de Água', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='r', linestyle='--', label='Superfície')
axes[1].axhline(y=-L_lençol, color='g', linestyle='--', label='Lençol Freático')
axes[1].axvline(x=theta_s, color='b', linestyle='--', label=f'θ_s = {theta_s}')
axes[1].axvline(x=theta_r, color='orange', linestyle='--', label=f'θ_r = {theta_r}')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print("\n💡 Interpretação:")
print("  • ψ torna-se mais negativo próximo à superfície (solo mais seco)")
print("  • θ diminui com a distância ao lençol freático")
print("  • A zona não-saturada se estende acima do lençol")

---

## 🔍 Seção 5: Análise de Sensibilidade

### Efeito da Profundidade do Lençol Freático

In [ ]:
# ============================================================
# ANÁLISE DE SENSIBILIDADE
# ============================================================

print("=" * 60)
print("ANÁLISE DE SENSIBILIDADE")
print("=" * 60)

# Variar profundidade do lençol
L_values = [1.0, 2.0, 3.0, 5.0, 10.0]  # m
psi_superficie = []
theta_superficie = []

for L_teste in L_values:
    # Resolver para esta profundidade
    N_teste = 31
    dz_teste = L_teste / (N_teste - 1)
    psi_teste = np.zeros(N_teste)
    psi_teste[-1] = 0.0
    
    for i in range(N_teste-2, -1, -1):
        psi_prev = psi_teste[i+1]
        psi_i = psi_prev - dz_teste
        
        for k in range(max_iter):
            K_i = K_de_psi(psi_i, theta_r, theta_s, alpha, n, K_sat, L_param)
            if K_i < 1e-15:
                K_i = 1e-15
            
            psi_novo = psi_prev + dz_teste * (q0 / K_i - 1)
            
            if abs(psi_novo - psi_i) < tol:
                psi_teste[i] = psi_novo
                break
            
            psi_i = psi_novo
    
    psi_superficie.append(psi_teste[0])
    theta_superficie.append(van_genuchten_theta(psi_teste[0], theta_r, theta_s, alpha, n)[0])

print(f"\n📊 Efeito da profundidade do lençol freático:")
print(f"{'L [m]':<10} {'ψ_sup [m]':<15} {'θ_sup [m³/m³]':<15}")
print("-" * 40)
for L_val, psi_val, theta_val in zip(L_values, psi_superficie, theta_superficie):
    print(f"{L_val:<10.1f} {psi_val:<15.3f} {theta_val:<15.3f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(L_values, psi_superficie, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Profundidade do Lençol, L [m]', fontsize=12)
axes[0].set_ylabel('ψ na Superfície [m]', fontsize=12)
axes[0].set_title('Potencial Matricial na Superfície', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(L_values, theta_superficie, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Profundidade do Lençol, L [m]', fontsize=12)
axes[1].set_ylabel('θ na Superfície [m³/m³]', fontsize=12)
axes[1].set_title('Conteúdo de Água na Superfície', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=theta_s, color='b', linestyle='--', label=f'θ_s = {theta_s}')
axes[1].axhline(y=theta_r, color='g', linestyle='--', label=f'θ_r = {theta_r}')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"\n💡 Interpretação:")
print(f"  • Quanto maior L, mais negativo ψ na superfície")
print(f"  • θ na superfície diminui com L (solo mais seco)")
print(f"  • Para L muito grande, θ → θ_r (solo residual)")

---

## 💡 Seção 6: Discussão e Conclusões

### Resultados Principais

1. **Fluxo compatível:** A infiltração $q_0 = 1,0 \times 10^{-6}$ m/s (86,4 mm/dia) é compatível com as propriedades do solo franco-arenoso, pois $q_0 < K_{sat}$.

2. **Perfil de umidade:** O conteúdo de água diminui exponencialmente com a distância ao lençol freático, refletindo a não-linearidade de $K(\theta)$.

3. **Sensibilidade:** A profundidade do lençol freático $L$ é o parâmetro mais influente no perfil de umidade. Um aumento de 3 m para 10 m reduz $\theta$ na superfície de $\approx 0,35$ para $\approx 0,08$ m³/m³.

### Aplicações Práticas

- **Agricultura:** O perfil de umidade determina a disponibilidade de água para as raízes das plantas.
- **Geotecnia:** A sucção matricial afeta a resistência ao cisalhamento do solo.
- **Hidrologia:** O modelo é essencial para estimar recarga de aquíferos e escoamento subterrâneo.

### Limitações

- **Regime permanente:** A solução assume fluxo constante, o que não é realista em eventos de chuva.
- **Solo homogêneo:** O modelo não considera estratificação ou anisotropia.
- **Histerese:** O modelo de van Genuchten não captura efeitos de histerese na curva de retenção.

---

## 🔄 Navegação

- [📚 Voltar ao Capítulo 6](../notebooks/06_Percolacao_Meio_Poroso.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**Estudo de Caso 6.1**  
Parte da coleção "Fenômenos de Transporte: Fundamentos e Modelagem Computacional"

</div>